# Universidad Tecnológica Metropolitana del Estado de Chile
## Ingeniería Civil en Ciencia de Datos
### INFB8090 — Computación Paralela y Distribuida

---

## LABORATORIO N° 1: Diagnóstico experimental de paralelismo

| Campo | Detalle |
|---|---|
| **Asignatura** | INFB8090 — Computación Paralela y Distribuida |
| **Sección** | 001 |
| **Fecha de desarrollo** | 02 de abril de 2026 |
| **Fecha de entrega** | 08 de abril de 2026 |
| **Docente** | Dr. Ing. Michael Miranda Sandoval |
| **Integrante(s)** | *[Nombre del estudiante]* |
| **Equipo** | *[Ej: Intel Core i7-11th, 16 GB RAM, Python 3.10, Ubuntu 22.04]* |

---
## Contexto y objetivo del laboratorio

El presente laboratorio constituye la primera aproximación experimental a la asignatura *Computación Paralela y Distribuida*. Su propósito no es aún implementar soluciones paralelas avanzadas, sino desarrollar una **mentalidad de diagnóstico riguroso** frente al rendimiento computacional.

Para tomar decisiones informadas sobre paralelización, es imprescindible:
1. **Medir** tiempos de ejecución con herramientas precisas.
2. **Identificar** la estructura interna del problema (dependencias entre tareas).
3. **Analizar** si existe independencia suficiente para justificar la distribución del trabajo.

Este laboratorio explora tres escenarios distintos:
- **Ejercicio 1:** Medición de una operación con dependencias acumulativas (sucesión de Fibonacci).
- **Ejercicio 2:** Comparación entre procesamiento iterativo y procesamiento por particiones sobre transformaciones de señales.
- **Ejercicio 3:** Simulación de un pipeline de preprocesamiento de datos geoespaciales por regiones, con decisión diagnóstica sobre la estrategia computacional óptima.

La pregunta central que guía todo el laboratorio es: **¿cuándo paralelo, cuándo distribuido y cuándo secuencial?**

---
## Ejercicio 1: Medición base sobre una tarea con dependencias acumulativas

### Descripción del problema

En lugar de la suma de cuadrados canónica, se trabaja aquí con el cálculo de la **sucesión de Fibonacci iterativa**. Esta operación es inherentemente secuencial porque cada término depende directamente del anterior: no es posible calcular `F(n)` sin conocer `F(n-1)` y `F(n-2)`. Esta propiedad la convierte en un caso de estudio ideal para establecer una línea base y argumentar la *imposibilidad estructural* de paralelizar ciertos algoritmos.

Se ejecutará la función con tres tamaños de entrada crecientes y se medirá el tiempo con `time.perf_counter()`, el reloj de mayor resolución disponible en Python para benchmarking.

In [ ]:
import time

# ---------------------------------------------------------
# Función: fibonacci_iterativo
# Calcula el n-ésimo número de Fibonacci de forma iterativa.
# Cada paso depende del resultado inmediatamente anterior,
# lo que crea una cadena de dependencias de datos.
# ---------------------------------------------------------
def fibonacci_iterativo(n):
    if n <= 1:
        return n
    anterior, actual = 0, 1
    for _ in range(2, n + 1):
        anterior, actual = actual, anterior + actual
    return actual


# Tamaños de entrada (número de términos a calcular)
entradas = [500_000, 2_000_000, 5_000_000]

print(f"{'Tamaño N':>12} | {'Tiempo (s)':>12} | {'Resultado (mod 10^9)':>22}")
print("-" * 54)

tabla_ej1 = []
for n in entradas:
    t0 = time.perf_counter()
    resultado = fibonacci_iterativo(n)
    t1 = time.perf_counter()
    duracion = t1 - t0
    # Mostramos el resultado módulo 10^9 para evitar imprimir un número enorme
    resultado_display = resultado % (10**9)
    tabla_ej1.append((n, duracion, resultado_display))
    print(f"{n:>12,} | {duracion:>12.6f} | {resultado_display:>22,}")

In [ ]:
import matplotlib.pyplot as plt

ns      = [r[0] for r in tabla_ej1]
tiempos = [r[1] for r in tabla_ej1]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ns, tiempos, marker='o', linewidth=2, color='steelblue', markersize=8)
ax.set_xlabel("Tamaño de entrada (n)", fontsize=12)
ax.set_ylabel("Tiempo de ejecución (s)", fontsize=12)
ax.set_title("Ejercicio 1 — Crecimiento lineal del tiempo en Fibonacci iterativo", fontsize=13)
ax.set_xticks(ns)
ax.set_xticklabels([f"{n:,}" for n in ns])
ax.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig("ej1_fibonacci_tiempos.png", dpi=120)
plt.show()
print("Gráfico guardado como ej1_fibonacci_tiempos.png")

### Interpretación y análisis — Ejercicio 1

**Crecimiento temporal observado:**  
El tiempo de ejecución escala de manera aproximadamente lineal con el tamaño de la entrada. Esto es coherente con la complejidad `O(n)` del algoritmo: cada término requiere exactamente una operación de suma y una reasignación de variables.

**¿Por qué esta tarea es estrictamente secuencial?**  
La secuencia de Fibonacci presenta una **dependencia de datos total entre iteraciones**: el valor del término `F(k)` no puede obtenerse hasta conocer `F(k-1)` y `F(k-2)`. No existe ninguna sub-tarea independiente que pueda asignarse a otro hilo o proceso sin comunicación previa. Intentar paralelizar esta función introduciría un costo de sincronización que superaría cualquier ganancia potencial.

**Relación con el concepto de "fracción paralelizable" (Ley de Amdahl):**  
Si la fracción secuencial obligatoria de un programa es alta (como en este caso, donde es prácticamente 100%), la Ley de Amdahl predice que la aceleración máxima posible tiende a 1, es decir, no hay beneficio real al agregar más procesadores.

**Conclusión del ejercicio:**  
Esta medición establece la línea base de referencia. Cualquier comparación futura con versiones paralelas debe contrastarse con estos tiempos para cuantificar el speedup real obtenido.

---
## Ejercicio 2: Comparación de variantes — procesamiento iterativo vs. procesamiento por particiones

### Descripción del problema

Se trabaja con una colección de valores que representan **amplitudes de señal discreta** (números de punto flotante). La tarea es aplicar una transformación matemática costosa a cada elemento: calcular la **energía normalizada** de cada muestra, definida como `(sin(x)² + cos(x)²) / (1 + |x|)`. Esta operación es independiente entre muestras, lo que la convierte en un candidato natural para paralelización.

Se comparan dos implementaciones secuenciales:
- **Variante A:** Iteración elemento a elemento sobre la lista completa.
- **Variante B:** Procesamiento dividido en ventanas (particiones) de tamaño fijo.

El objetivo no es aún paralelizar, sino **evidenciar la independencia estructural** entre particiones.

In [ ]:
import time
import math
import random

random.seed(2026)

# Generar señal: 600,000 valores flotantes en [-π, π]
N_SENIAL = 600_000
senial = [random.uniform(-math.pi, math.pi) for _ in range(N_SENIAL)]

# ---------------------------------------------------------
# Transformación: energía normalizada de cada muestra
# La función es costosa por involucrar sin, cos y división.
# ---------------------------------------------------------
def transformar_muestra(x):
    return (math.sin(x)**2 + math.cos(x)**2) / (1.0 + abs(x))


# ---- VARIANTE A: iteración elemento a elemento ----
def procesar_iterativo(datos):
    acumulado = 0.0
    for muestra in datos:
        acumulado += transformar_muestra(muestra)
    return acumulado


# ---- VARIANTE B: procesamiento por ventanas de tamaño fijo ----
def procesar_por_ventanas(datos, tam_ventana=60_000):
    total = 0.0
    n = len(datos)
    for inicio in range(0, n, tam_ventana):
        ventana = datos[inicio : inicio + tam_ventana]
        subtotal = 0.0
        for muestra in ventana:
            subtotal += transformar_muestra(muestra)
        total += subtotal
    return total


# --- Medición Variante A ---
t0 = time.perf_counter()
resultado_a = procesar_iterativo(senial)
tiempo_a = time.perf_counter() - t0

# --- Medición Variante B ---
t0 = time.perf_counter()
resultado_b = procesar_por_ventanas(senial)
tiempo_b = time.perf_counter() - t0

print(f"VARIANTE A — Iterativo")
print(f"  Resultado : {resultado_a:.6f}")
print(f"  Tiempo    : {tiempo_a:.6f} s")
print()
print(f"VARIANTE B — Por ventanas")
print(f"  Resultado : {resultado_b:.6f}")
print(f"  Tiempo    : {tiempo_b:.6f} s")
print()
diferencia = abs(resultado_a - resultado_b)
print(f"Diferencia numérica entre variantes: {diferencia:.2e} (errores de punto flotante esperados)")

In [ ]:
import matplotlib.pyplot as plt

etiquetas  = ['Variante A\n(Iterativo)', 'Variante B\n(Por ventanas)']
tiempos_b  = [tiempo_a, tiempo_b]
colores    = ['#e07b54', '#5b8db8']

fig, ax = plt.subplots(figsize=(7, 4))
barras = ax.bar(etiquetas, tiempos_b, color=colores, width=0.4, edgecolor='black')

for barra, t in zip(barras, tiempos_b):
    ax.text(barra.get_x() + barra.get_width()/2, barra.get_height() + 0.005,
            f"{t:.4f} s", ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel("Tiempo de ejecución (s)", fontsize=12)
ax.set_title("Ejercicio 2 — Comparación de variantes de procesamiento", fontsize=13)
ax.set_ylim(0, max(tiempos_b) * 1.25)
ax.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig("ej2_variantes_procesamiento.png", dpi=120)
plt.show()
print("Gráfico guardado como ej2_variantes_procesamiento.png")

### Interpretación y análisis — Ejercicio 2

**Observación de tiempos:**  
Ambas variantes producen resultados numéricamente equivalentes (la pequeña diferencia se debe a errores de redondeo acumulativo en punto flotante, propios del orden de suma). Los tiempos son cercanos entre sí porque ambas versiones siguen siendo **secuenciales**: el intérprete de Python ejecuta cada ventana una tras otra.

**¿Qué revela la versión por ventanas?**  
La partición en ventanas expone la **independencia funcional** entre subconjuntos: el resultado de la ventana 1 no depende en absoluto de la ventana 2, ni viceversa. Cada ventana es una **unidad de trabajo autónoma**.

**¿Tendría sentido paralelo bajo qué condiciones?**  
Sí, bajo las siguientes condiciones:
- El costo de lanzar los hilos/procesos (`overhead`) debe ser menor que el tiempo de procesamiento de cada ventana. Para 60,000 elementos con operaciones trigonométricas, este umbral se cumple.
- La función `transformar_muestra` es **libre de efectos secundarios** (no modifica estado compartido), lo que la hace segura para ejecución concurrente.
- Se dispone de múltiples núcleos físicos. En Python, el GIL (*Global Interpreter Lock*) impediría beneficios reales con `threading`, pero `multiprocessing` o bibliotecas como `concurrent.futures` sí podrían explotarlos.

**Conclusión del ejercicio:**  
La estructura de ventanas independientes es el patrón central del paralelismo de datos. La Variante B no es más rápida aún (ambas son secuenciales), pero su diseño ya está listo para ser escalado horizontalmente asignando cada ventana a un worker independiente.

---
## Ejercicio 3: Caso aplicado — Pipeline de preprocesamiento geoespacial por regiones

### Descripción del problema

Se simula un escenario real de ciencia de datos: el preprocesamiento de **registros de temperatura diaria** provenientes de distintas regiones geográficas de Chile. Cada región corresponde a un lote de datos independiente.

Para cada región se calcula un conjunto de estadísticos descriptivos y un índice de variabilidad climática: la **desviación estándar normalizada** (coeficiente de variación). Este pipeline es análogo a tareas reales como la normalización de features por grupo, el cálculo de estadísticos por partición en bases de datos distribuidas, o la validación cruzada estratificada.

La pregunta diagnóstica es: **¿esta carga de trabajo justifica paralelismo local, distribución entre nodos, o basta con la estrategia secuencial?**

In [ ]:
import time
import random
import math
import statistics

random.seed(42)

# Nombres de regiones (simuladas, representan 15 zonas climáticas)
REGIONES = [
    "Arica y Parinacota", "Tarapacá", "Antofagasta", "Atacama",
    "Coquimbo", "Valparaíso", "Metropolitana", "O'Higgins",
    "Maule", "Ñuble", "Biobío", "La Araucanía",
    "Los Ríos", "Los Lagos", "Aysén"
]

# Generar datos: cada región tiene 150,000 registros de temperatura (°C)
# Las temperaturas varían según la latitud simulada de cada región
def generar_lote_regional(nombre_region, n=150_000):
    indice = REGIONES.index(nombre_region)
    # Temperatura base decrece hacia el sur
    temp_base = 30.0 - indice * 1.5
    # Variabilidad también cambia por región
    variabilidad = 3.0 + indice * 0.3
    return [random.gauss(temp_base, variabilidad) for _ in range(n)]

print("Generando lotes de datos por región...")
t_gen0 = time.perf_counter()
lotes_regionales = {region: generar_lote_regional(region) for region in REGIONES}
t_gen1 = time.perf_counter()
print(f"  → {len(REGIONES)} regiones × 150,000 registros generados en {t_gen1 - t_gen0:.3f} s")
print(f"  → Total de registros: {len(REGIONES) * 150_000:,}")

In [ ]:
# ---------------------------------------------------------
# Pipeline de preprocesamiento por región
# Para cada lote calcula: media, mediana, desvío estándar,
# percentil 5, percentil 95 y coeficiente de variación.
# ---------------------------------------------------------
def preprocesar_region(nombre, datos):
    media    = statistics.fmean(datos)
    mediana  = statistics.median(datos)
    desvio   = statistics.stdev(datos)
    datos_s  = sorted(datos)
    n        = len(datos_s)
    p5       = datos_s[int(0.05 * n)]
    p95      = datos_s[int(0.95 * n)]
    cv       = (desvio / media) * 100 if media != 0 else float('nan')
    return {
        "region" : nombre,
        "media"  : round(media, 3),
        "mediana": round(mediana, 3),
        "desvio" : round(desvio, 3),
        "p5"     : round(p5, 3),
        "p95"    : round(p95, 3),
        "cv_pct" : round(cv, 3)
    }


# --- Ejecución secuencial del pipeline ---
t_proc0 = time.perf_counter()
resultados_regiones = []
for region, lote in lotes_regionales.items():
    resumen = preprocesar_region(region, lote)
    resultados_regiones.append(resumen)
t_proc1 = time.perf_counter()
tiempo_pipeline = t_proc1 - t_proc0


# --- Presentación de resultados ---
print(f"\n{'Región':<22} {'Media':>8} {'Mediana':>9} {'Desvío':>8} {'P5':>8} {'P95':>8} {'CV%':>7}")
print("-" * 76)
for r in resultados_regiones:
    print(f"{r['region']:<22} {r['media']:>8.3f} {r['mediana']:>9.3f} "
          f"{r['desvio']:>8.3f} {r['p5']:>8.3f} {r['p95']:>8.3f} {r['cv_pct']:>7.3f}")

print(f"\nTiempo total del pipeline: {tiempo_pipeline:.6f} s")
print(f"Tiempo promedio por región: {tiempo_pipeline / len(REGIONES):.6f} s")

In [ ]:
import matplotlib.pyplot as plt

nombres = [r['region'] for r in resultados_regiones]
medias  = [r['media']  for r in resultados_regiones]
desvios = [r['desvio'] for r in resultados_regiones]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: temperatura media por región
axes[0].barh(nombres, medias, color='tomato', edgecolor='black')
axes[0].set_xlabel("Temperatura media (°C)", fontsize=11)
axes[0].set_title("Temperatura media simulada por región", fontsize=12)
axes[0].invert_yaxis()
axes[0].grid(axis='x', linestyle='--', alpha=0.6)

# Gráfico 2: desvío estándar (variabilidad climática)
axes[1].barh(nombres, desvios, color='steelblue', edgecolor='black')
axes[1].set_xlabel("Desvío estándar (°C)", fontsize=11)
axes[1].set_title("Variabilidad climática por región", fontsize=12)
axes[1].invert_yaxis()
axes[1].grid(axis='x', linestyle='--', alpha=0.6)

plt.suptitle("Ejercicio 3 — Pipeline de preprocesamiento geoespacial", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("ej3_pipeline_regiones.png", dpi=120, bbox_inches='tight')
plt.show()
print("Gráfico guardado como ej3_pipeline_regiones.png")

### Análisis diagnóstico — Ejercicio 3

#### Descripción del problema en términos de unidades de trabajo

El pipeline consiste en **15 unidades de trabajo completamente independientes**, una por región. Cada unidad recibe como entrada su lote exclusivo de 150,000 registros y produce como salida un conjunto de estadísticos. No existe ningún flujo de datos entre regiones: el procesamiento de Arica no afecta en absoluto al de Magallanes.

#### ¿Secuencial, paralela local o distribuida?

**Decisión diagnóstica: paralelismo local con `multiprocessing` o `concurrent.futures`.**

Las razones que fundamentan esta elección son:

1. **Independencia total entre tareas:** cada región es una unidad de trabajo sin dependencias de datos externas. Este es el requisito fundamental para paralelizar sin necesidad de sincronización.

2. **Volumen moderado por unidad:** 150,000 registros por región son suficientes para que el costo de lanzar un subproceso sea negligible frente al tiempo de cómputo de `statistics.stdev` y `sorted()`. Sin embargo, el volumen total de 2.25 millones de registros no justifica por sí solo trasladar la carga a un clúster distribuido.

3. **Costo computacional no trivial:** las operaciones de ordenamiento (`O(n log n)`) y cálculo de desvío estándar sobre listas grandes generan una carga real que puede aprovechar múltiples núcleos físicos.

4. **Necesidad de coordinación mínima:** solo se necesita recolectar los 15 diccionarios de resultados al final, lo que representa una comunicación de datos despreciable.

5. **¿Por qué no distribuido?** Un sistema distribuido (Spark, Dask en modo clúster) estaría justificado si los lotes fueran de millones de registros cada uno, si los datos residieran en sistemas de almacenamiento remoto, o si el pipeline incluyera transformaciones que requieran coordinación global (joins, shuffles). Para el volumen actual, el overhead de red y serialización de un sistema distribuido superaría las ganancias.

#### Estimación teórica del speedup (Ley de Amdahl)

Si la fracción paralelizable del pipeline es aproximadamente el 95% (existe un pequeño overhead de inicialización secuencial), con 8 núcleos la Ley de Amdahl predice:

$$S(8) = \frac{1}{(1 - 0.95) + \frac{0.95}{8}} = \frac{1}{0.05 + 0.119} \approx 5.9\times$$

Esto significa que en un equipo de 8 núcleos podríamos esperar reducir el tiempo del pipeline en casi 6 veces, procesando las 15 regiones en aproximadamente 6 grupos paralelos en lugar de secuencialmente.

---
## Conclusiones y Reflexión Diagnóstica Final

### Síntesis de resultados

| Ejercicio | Tarea | Naturaleza | ¿Paralelizable? | Estrategia óptima |
|---|---|---|---|---|
| 1 | Fibonacci iterativo | Dependencias en cadena | ❌ No | Secuencial |
| 2 | Transformación de señal por ventanas | Independencia elemental | ✅ Sí | Paralelo local (multiprocessing) |
| 3 | Pipeline geoespacial por regiones | Independencia por partición | ✅ Sí | Paralelo local; distribuido solo si escala masivamente |

---

### Reflexión diagnóstica

**¿Qué características debe tener un problema para que el paralelismo resulte pertinente?**  
Un problema justifica paralelización cuando: (1) puede descomponerse en unidades de trabajo independientes o con dependencias mínimas, (2) el costo computacional de cada unidad es suficientemente alto para amortizar el overhead de lanzar procesos/hilos, y (3) el volumen total de datos hace que la ejecución secuencial sea un cuello de botella medible. La independencia estructural es el requisito no negociable: sin ella, la sincronización destruye cualquier ganancia.

**¿Por qué no basta con que un programa "demore" para concluir que debe paralelizarse?**  
Un programa puede demorar por razones completamente ajenas a su estructura computacional: acceso a disco lento, latencia de red, carga de datos en memoria, o simplemente dependencias secuenciales irrompibles como en el caso de Fibonacci. Paralelizar un cuello de botella de I/O no reduce el tiempo de espera del disco. Paralelizar una cadena de dependencias de datos agrega overhead sin reducir el tiempo de cómputo. Por eso, la medición y el análisis estructural del problema deben preceder a cualquier decisión de optimización.

**¿En qué tipos de tareas de ciencia de datos el paralelismo o la distribución aportarían mayor valor?**  
Los candidatos más claros son: entrenamiento de múltiples modelos con distintos hiperparámetros (gridsearch paralelo), preprocesamiento de datasets particionados (por fecha, región o categoría), generación de features independientes sobre millones de filas, evaluación de métricas en validación cruzada estratificada, y simulaciones Monte Carlo. Todos comparten la propiedad de ser *embarrassingly parallel*: las unidades de trabajo no necesitan comunicarse entre sí.

**¿Qué aprendí sobre la importancia de medir antes de optimizar?**  
Este laboratorio deja en evidencia que la intuición no es suficiente. Sin medir la línea base secuencial, no es posible cuantificar el speedup real de una implementación paralela, ni detectar si el tiempo se concentra en la fracción paralelizable o en overheads inevitables. Optimizar sin medir es una inversión ciega: se puede gastar horas paralelizando una función que no es el cuello de botella real del sistema. El profiling experimental es el punto de partida obligatorio de toda ingeniería de rendimiento seria.

---
## Referencias

- Documentación oficial de Python 3. https://docs.python.org/3/
- Python Standard Library: `time`. https://docs.python.org/3/library/time.html
- Python Standard Library: `statistics`. https://docs.python.org/3/library/statistics.html
- Jupyter Documentation. https://docs.jupyter.org/
- Amdahl, G. M. (1967). *Validity of the single processor approach to achieving large scale computing capabilities*. AFIPS Spring Joint Computer Conference.
- Material de cátedra de la asignatura *Computación Paralela y Distribuida*, Semana 1 — UTEM.